# Practical Problems 2 — Dynamic Programming
## TL;DR — Plain English Introduction

**What is dynamic programming?** DP = "remember the answers to subproblems so you don't recompute them." It turns exponential-time recursive solutions into polynomial-time ones.

**The three-step DP recipe:**
1. **Define the state**: What does `dp[i]` mean?
2. **Write the recurrence**: How does `dp[i]` depend on smaller states?
3. **Set the base case**: What is `dp[0]` or `dp[1]`?

**Classic example — Fibonacci without DP:** O(2^n) (recomputes everything)
**With DP (memoization):** O(n) — store results in a dict

**No Python background?**
```python
# Memoization = cache function results
from functools import lru_cache
@lru_cache(maxsize=None)  # add this decorator
def fib(n):              # and any recursive function
    if n <= 1: return n  # becomes fast automatically
    return fib(n-1) + fib(n-2)
```

**What you'll solve:**
- HackerRank: Fibonacci Modified, Coin Change, LCS ("Common Child"), LIS
- Rosalind: FIB/FIBD (rabbit populations), RNA secondary structure (PMCH, CAT)

**Learning path:** 00/01 (Python recursion) → This notebook → 08/03 (Graphs) → 01/05 (Phylogeny)

## Beginner Teaching Frame

**Who should fully work through this notebook:** every student. This is where fluency is built.

**How to study it on a first pass:**
- try each problem before reading the answer
- keep a timer for at least one section
- write the complexity and the biological interpretation after each solution

**Common traps:** reading solution code too early, confusing recognition with mastery, and skipping timed practice because the answers are available.


## 📖 Prerequisites & Cross-References

**This is a practice notebook.** It tests concepts from earlier modules.

**If you get stuck, review:**
- DNA/string problems → `01_sequence_analysis/01_alignment_algorithms.ipynb`
- DP problems → `01/01` (Needleman-Wunsch uses dynamic programming)
- Graph problems → `06_structural_ml_gnns/01_structure_ml.ipynb`
- Statistics → `00_python_ml_basics/05_hackerrank_statistics_ml_tracks.ipynb`
- Alignment/phylogeny → `01/04` and `01/05`

## Canonical Resource Upgrade

Use these as practice companions:
- [HackerRank](https://www.hackerrank.com/domains)
- [Rosalind](https://rosalind.info/problems/list-view/)
- [Big-O Cheat Sheet](https://www.bigocheatsheet.com/)


# Practical 2 — Dynamic Programming & drug discovery companies

## What This Notebook Is

DP is the single most tested algorithmic concept in bioinformatics interviews. Both HackerRank and Rosalind test it heavily. Here every DP concept is shown **in context** — first the pure algorithmic version, then the biological application.

**Tags**: `[HR]` = HackerRank | `[R:ID]` = Rosalind problem ID

## Learning Objectives
- [ ] Implement memoized recursion and bottom-up DP
- [ ] Solve Fibonacci variants (rabbits, wabbits with death)
- [ ] Implement coin change, subset sum, longest subsequences
- [ ] Know when to use top-down vs bottom-up
- [ ] Handle large numbers with modular arithmetic

## The DP Template
```
1. Define subproblem: dp[i] = ?
2. Base case: dp[0] = ?, dp[1] = ?
3. Recurrence: dp[i] = f(dp[i-1], dp[i-2], ...)
4. Answer: dp[n]
```

## 🗺️ Prerequisites & Learning Path

| | |
|--|--|
| 🔑 Prerequisites | 00/01 (Python fundamentals), 01/01 (basic sequences for biological DP examples) |
| 📍 You Are Here | Module 08/02 — Dynamic Programming |
| ➡️ Next Steps | 08/03 (graphs and genome assembly) |
| 🏁 Goal | Implement Nussinov RNA folding, LCS, and coin change from scratch; recognize DP patterns in interview problems |

### 🆕 From Scratch? Start Here:
1. [LeetCode DP study plan](https://leetcode.com/study-plan/dynamic-programming/) — 25 starter problems
2. [Rosalind FIB problem](https://rosalind.info/problems/fib/) — entry-level DP with biology context
3. 00/01 (this repo) — Python fundamentals
4. This notebook — dynamic programming
**Time:** 10–15h | **Difficulty:** ⭐⭐⭐⭐

### 🔗 Cross-References
- Builds on: 00/01 (Python), 01/01 (biological sequences as DP inputs), 08/01 (string DP — edit distance, LCS)
- Used by: 08/05 (sequence alignment is applied DP), 07/01–07/04 (AF3 uses DP concepts in energy functions)

In [ ]:
# DP Section 1 — Fibonacci + Rosalind FIB/FIBD
from functools import lru_cache

# --- Rosalind FIB (Fibonacci rabbits) ---
def fib_rabbits(n, k):
    """n months, k offspring per pair. F(n) = F(n-1) + k*F(n-2)"""
    a, b = 1, 1
    for _ in range(n-2):
        a, b = b, k*a + b
    return b

print(f"FIB(5,3) = {fib_rabbits(5,3)}")  # 19

# --- Rosalind FIBD (mortal Fibonacci) ---
def mortal_fib(n, m):
    """Rabbits live m months. DP with array of age classes."""
    ages = [0] * m
    ages[0] = 1  # 1 newborn pair
    for _ in range(n-1):
        new_born = sum(ages[1:])  # all adults breed
        ages = [new_born] + ages[:-1]  # shift right, drop oldest
    return sum(ages)

print(f"FIBD(6,3) = {mortal_fib(6,3)}")  # 4

# --- HackerRank: Fibonacci Modified ---
def fib_modified(a, b, n):
    """t(i) = t(i-2) + t(i-1)^2"""
    seq = [a, b]
    for i in range(2, n):
        seq.append(seq[-2] + seq[-1]**2)
    return seq[n-1]

print(f"Fib modified t(1)=0,t(2)=1,n=5: {fib_modified(0,1,5)}")

> **Expected output:** Fibonacci variant results: `FIB(5,3) = 19`, `FIBD(6,3) = 4`, and a modified Fibonacci value  
> If you see this, your code is working correctly.  
> If you see an error, check the Troubleshooting section or re-run the cell.

> **Why this code:** This function processes biological data, converting raw sequences into a format our analysis can use.

In [ ]:
# DP Section 2 — LCS + LIS
import numpy as np

# --- LCS / Rosalind LCSM ---
def lcs(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0]*(n+1) for _ in range(m+1)]
    # --- Loop over range(1,m+1) ---
    for i in range(1,m+1):
        # --- Loop over range(1,n+1) ---
        for j in range(1,n+1):
            if s1[i-1]==s2[j-1]:
                dp[i][j] = dp[i-1][j-1]+1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    # Traceback
    result = []
    i, j = m, n
    # --- While loop ---
    while i > 0 and j > 0:
        if s1[i-1]==s2[j-1]:
            result.append(s1[i-1]); i-=1; j-=1
        elif dp[i-1][j] > dp[i][j-1]:
            i -= 1
        else:
            j -= 1
    return ''.join(reversed(result))

print("LCS of 'AACCTTGG' and 'ACACTGTGA':", lcs("AACCTTGG","ACACTGTGA"))

# --- LIS (Longest Increasing Subsequence) O(n log n) ---
import bisect
# --- lis() function ---
def lis(seq):
    tails = []
    # --- Loop over seq ---
    for x in seq:
        pos = bisect.bisect_left(tails, x)
        if pos == len(tails):
            tails.append(x)
        else:
            tails[pos] = x
    return len(tails)

arr = [3,1,4,1,5,9,2,6,5,3,5]
print(f"LIS of {arr}: {lis(arr)}")

In [ ]:
# DP Section 3 — Coin change + Knapsack
# --- Coin change (minimum coins) ---
def min_coins(amount, coins):
    dp = [float('inf')] * (amount + 1)
    dp[0] = 0
    # --- Loop over range(1, amount+1) ---
    for i in range(1, amount+1):
        # --- Loop over coins ---
        for c in coins:
            if c <= i and dp[i-c]+1 < dp[i]:
                dp[i] = dp[i-c]+1
    return dp[amount] if dp[amount] != float('inf') else -1

print(f"Min coins for 11 with [1,5,6,9]: {min_coins(11,[1,5,6,9])}")

# --- Rosalind PPER (partial permutations) ---
MOD = 1_000_000
# --- partial_permutations() function ---
def partial_permutations(n, k, mod=1_000_000):
    result = 1
    # --- Loop over range(n, n-k, -1) ---
    for i in range(n, n-k, -1):
        result = (result * i) % mod
    return result

print(f"P(21,7) mod 1000000 = {partial_permutations(21,7)}")

# --- Rosalind SIGN (signed permutations) ---
from itertools import permutations
# --- signed_permutations() function ---
def signed_permutations(n):
    # --- Imports ---
    from itertools import product
    result = []
    # --- Loop over permutations(range(1, n+1)) ---
    for perm in permutations(range(1, n+1)):
        # --- Loop over product([-1,1], repeat=n) ---
        for signs in product([-1,1], repeat=n):
            result.append(tuple(s*p for s,p in zip(signs,perm)))
    return result

sp = signed_permutations(2)
print(f"Signed permutations of n=2: {len(sp)} total")
# --- Loop over sp[ ---
for p in sp[:4]:
    print(f"  {p}")

In [ ]:
# DP Section 4 — RNA secondary structure (Nussinov)
import numpy as np

# --- nussinov() function ---
def nussinov(rna, min_loop=4):
    """Nussinov algorithm for RNA secondary structure (max base pairs)."""
    n = len(rna)
    # Initialize tensor
    dp = np.zeros((n, n), dtype=int)
    complement = {'A':'U','U':'A','G':'C','C':'G'}

    # --- Loop over range(min_loop+1, n) ---
    for gap in range(min_loop+1, n):
        # --- Loop over range(n - gap) ---
        for i in range(n - gap):
            j = i + gap
            options = [dp[i+1][j], dp[i][j-1]]
            if complement.get(rna[i]) == rna[j]:
                options.append(dp[i+1][j-1] + 1)
            # Bifurcation
            for k in range(i+1, j):
                options.append(dp[i][k] + dp[k+1][j])
            dp[i][j] = max(options)

    # Traceback
    def traceback(i, j):
        if i >= j: return []
        if dp[i][j] == dp[i+1][j]:
            return traceback(i+1, j)
        if dp[i][j] == dp[i][j-1]:
            return traceback(i, j-1)
        # Conditional check
        if complement.get(rna[i]) == rna[j] and dp[i][j] == dp[i+1][j-1]+1:
            return [(i+1, j+1)] + traceback(i+1, j-1)
        # --- Loop over range(i+1, j) ---
        for k in range(i+1, j):
            if dp[i][j] == dp[i][k] + dp[k+1][j]:
                return traceback(i, k) + traceback(k+1, j)
        return []

    rna_seq = "AGUCAUUGCCAUGGCUAGUCUA"
    # Initialize tensor
    dp2 = np.zeros((len(rna_seq), len(rna_seq)), dtype=int)
    # Just show the max pairs for the input
    pairs = traceback(0, n-1)
    return dp[0][n-1], pairs

score, pairs = nussinov("AGUCAUUGCCAUGGCUAGUCUA")
print(f"Max base pairs: {score}")
print(f"Base pairs: {pairs[:5]}...")

> ✋ **Try it yourself:** Before running the next cell, predict what the output will be. Then run it and check. If you were wrong, re-read the code above.

In [ ]:
# DP Section 5 — Edit distance + Candies (HackerRank)
# --- Edit distance (Levenshtein) ---
def edit_distance(s1, s2):
    m, n = len(s1), len(s2)
    dp = list(range(n+1))
    # --- Loop over range(1, m+1) ---
    for i in range(1, m+1):
        prev = dp[:]
        dp[0] = i
        # --- Loop over range(1, n+1) ---
        for j in range(1, n+1):
            if s1[i-1] == s2[j-1]:
                dp[j] = prev[j-1]
            else:
                dp[j] = 1 + min(prev[j], dp[j-1], prev[j-1])
    return dp[n]

pairs = [("PLEASANTLY","MEANLY"), ("ACGT","AGT"), ("kitten","sitting")]
# --- Loop ---
for s1, s2 in pairs:
    print(f"edit_dist('{s1}','{s2}') = {edit_distance(s1,s2)}")

# --- HackerRank: Candies ---
def candies(ratings):
    """Min candies: each child gets ≥1 and more than neighbor if higher rating."""
    n = len(ratings)
    c = [1] * n
    # --- Loop over range(1, n) ---
    for i in range(1, n):
        if ratings[i] > ratings[i-1]:
            c[i] = c[i-1] + 1
    # --- Loop over range(n-2, -1, -1) ---
    for i in range(n-2, -1, -1):
        if ratings[i] > ratings[i+1]:
            c[i] = max(c[i], c[i+1]+1)
    return sum(c)

print(f"Candies for [1,2,2]: {candies([1,2,2])}")
print(f"Candies for [4,3,2,1]: {candies([4,3,2,1])}")

# Resources line
print("Key DP patterns: 1D DP, 2D table, interval DP, bitmask DP")

## Visualizing DP Time Complexity

In [ ]:
import numpy as np, matplotlib.pyplot as plt
n = np.arange(10, 1001, 10)
brute = 2**np.minimum(n, 30)  # Cap for display
quadratic = n**2
nlogn = n * np.log2(n)
plt.semilogy(n, quadratic, 'r-', label='O(n^2) -- DP', lw=2)
plt.semilogy(n, nlogn, 'g-', label='O(n log n) -- Divide & Conquer', lw=2)
plt.xlabel('Input Size n'); plt.ylabel('Operations (log scale)')
plt.title('Time Complexity: Why DP Beats Brute Force')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()  # transition probability (complement of 0.7)
print('DP trades space for time: O(n^2) table fills replace exponential recursion.')

## 📚 Resources

### 1️⃣ Theory — Foundations & Math
- **Optimal substructure**: solution to problem contains optimal solutions to subproblems — necessary condition for DP
- **Overlapping subproblems**: same subproblems appear multiple times — distinguishes DP from divide-and-conquer
- **Memoization vs tabulation**: top-down recursive + cache vs bottom-up iterative table — same asymptotic complexity, different constants
- **Nussinov algorithm**: RNA secondary structure DP — N(i,j) = max(N(i+1,j-1)+1, max_k(N(i,k)+N(k+1,j))) — O(N³) time
- **McCaskill partition function**: sum over all RNA structures with Boltzmann weights — gives base pair probabilities

### 2️⃣ Must-Have Popular Resources
- 📘 CLRS ch. 15 — DP with rod cutting, LCS, optimal BST — full formal treatment
- 📘 Dynamic Programming for Coding Interviews (Meenakshi & Kaur) — interview-focused patterns
- 🎓 LeetCode DP study plan — 75 problems organized by pattern (1D, 2D, interval, tree, bitmask)
- 🎓 Algorithms (Sedgewick) — concise DP chapter with Java visualizations (pseudocode transferable)
- ⭐ GitHub [TheAlgorithms/Python](https://github.com/TheAlgorithms/Python) — DP section with 30+ implementations

### 3️⃣ Practicals — Hands-On
- 💻 HackerRank: Fibonacci Modified, Max Array Sum, Abbreviation, Candies, Decibinary Numbers
- 💻 Rosalind: FIB, FIBD (mortality), PMCH (perfect matchings), CAT (Catalan numbers), PPER, SIGN (permutations)
- 💻 Classic LeetCode: Coin Change, LCS, Longest Increasing Subsequence, Edit Distance, Burst Balloons
- 💻 Implement Nussinov RNA folding from scratch — input: RNA sequence → output: max base-pair structure + dot-bracket notation
- 📊 Kaggle: [Algorithm optimization competitions](https://www.kaggle.com/search?q=dynamic+programming)

### 4️⃣ Real-World Problems
- 🏭 ViennaRNA (Lorenz et al.): industrial-standard RNA structure prediction using DP + thermodynamics
- 🏭 GATK HaplotypeCaller: sequence DP in variant calling (pair-HMM for read likelihood)
- 🏭 Protein energy minimization: DP over torsion angle space in Rosetta side-chain packing
- 📊 Dataset: [RNA secondary structure datasets](https://www.rnasoft.ca/strand/) — STRAND database for benchmarking
- 🔬 Application: mRNA vaccine codon optimization uses DP for maximum CAI scoring

### 5️⃣ Interview Tips
- ❓ Must know: Write LCS recurrence from scratch — `dp[i][j] = dp[i-1][j-1]+1 if match, else max(dp[i-1][j], dp[i][j-1])`
- ❓ Must know: Why RNA secondary structure is O(N³) — three nested loops over all intervals (i,j) and split points k
- ❓ Must know: Coin change bottom-up — `dp[i] = min(dp[i-c]+1 for c in coins if c<=i)`, `dp[0]=0`, rest = ∞
- ⚠️ Common mistake: Forgetting base cases — always write out dp[0] and dp[i][0] before the main recurrence
- 💡 Pro tip: For interval DP problems, always draw the i<j constraint as a triangle and fill bottom-up diagonally

### 6️⃣ Hot Industry Topics
- 🔥 Trending: [EternaFold](https://github.com/eternagame/EternaFold) — neural + DP hybrid RNA structure prediction (better than pure DP)
- 🔥 Trending: [ViennaRNA/ViennaRNA](https://github.com/ViennaRNA/ViennaRNA) — canonical DP package, now with ML extensions
- 🔥 Trending: AlphaFold3 diffusion replaces DP for proteins — but DP remains dominant for RNA
- 🚀 Build this: A full Nussinov RNA folder with backtracking that outputs dot-bracket notation and a circular arc diagram visualization

In [ ]:
# Resources for 08/02
print("KEY RESOURCES — Module 08/02: Dynamic Programming")
print()
refs = [
    "Rosalind DP problems: FIB, FIBD, LCSM, LGIS, PMCH, CAT, PPER, SIGN",
    "HackerRank DP: Fibonacci Modified, Coin Change, Candies",
    "Cormen et al. CLRS Chapter 15 — Dynamic Programming",
    "Competitive Programmer's Handbook — Antti Laaksonen (free PDF)",
    "InterviewBit DP problems",
    "Time complexity: most DP O(n^2) or O(n^3) for 2D tables",
]
for r in refs:
    print(f"  {r}")

print()
print("DP CHEAT SHEET:")
patterns = {
    "1D DP":       "dp[i] depends on dp[i-1], dp[i-2]... (Fibonacci, coins)",
    "2D DP":       "dp[i][j] depends on neighbors (LCS, edit distance, alignment)",
    "Interval DP": "dp[i][j] = opt over k in (i,j) (RNA struct, matrix chain)",
    "Tree DP":     "dp[v] depends on dp[children] (tree problems)",
}
for name, desc in patterns.items():
    print(f"  {name}: {desc}")

## Interview Cheat Sheet — DP

| Problem | Recurrence | Space | Time |
|---------|-----------|-------|------|
| Fibonacci | `f(n) = f(n-1) + f(n-2)` | O(1) | O(n) |
| LCS length | `dp[i][j] = dp[i-1][j-1]+1 or max(left,up)` | O(mn) | O(mn) |
| Edit distance | `dp[i][j] = 1+min(insert,delete,replace)` | O(min(m,n)) | O(mn) |
| LIS | Patience sort with bisect | O(n) | O(n log n) |
| Coin change | `dp[i] += dp[i-coin]` for each coin | O(n) | O(n × coins) |
| Max non-adj | `dp[i] = max(dp[i-1], dp[i-2]+arr[i])` | O(1) | O(n) |

**Top 3 mistakes**:
1. Forgetting modular arithmetic for large outputs (`% 10**9+7`)
2. Using recursion without `@lru_cache` → TLE (exponential time)
3. Off-by-one in DP table size (need `n+1` rows/cols for length-n strings)

## Real World Problems

### Problem 1: Protein Sequence DP
Given 10 protein sequences, find the longest common subsequence and compute edit distance between all pairs. Which two sequences are most similar? Most different?

**Resources**: [ProteinGym benchmark (HuggingFace)](https://huggingface.co/datasets/OATML-Markslab/ProteinGym_Substitutions) | [BioPython pairwise alignment](https://github.com/biopython/biopython)

### Problem 2: RNA Structure Prediction
Implement the Nussinov algorithm fully (with traceback to recover the actual pairing). Test on tRNA sequences from the Rfam database.

**Resources**: [Rfam RNA families (HuggingFace)](https://huggingface.co/datasets/Rfam/sequences) | [Nussinov original paper](https://pubmed.ncbi.nlm.nih.gov/6029570/)

### Problem 3: Rosalind DP Track
Complete these Rosalind DP problems in order: FIB → FIBD → LGIS → PPER → SIGN → CAT → MOTZ → NKEW → MMCH

**Resources**: [rosalind.info DP problems](https://rosalind.info/problems/list-view/) | [HackerRank DP challenges](https://www.hackerrank.com/domains/algorithms?filters%5Bsubdomains%5D%5B%5D=dynamic-programming)

## Mastery Check

Before moving on, make sure you can:
1. solve representative problems without looking up the answer
2. state the time complexity correctly
3. explain why your solution is correct
4. identify which earlier notebook you should revisit if a problem still feels opaque


---
## 🐛 Debug Exercise — Dynamic Programming Bugs

Find and fix **3 bugs** in this DP implementation.

<details><summary>Show bugs</summary>

**Bug 1:** Fibonacci with memoization: `memo[n] = fib(n-1) + fib(n-2)` is correct but `memo` dictionary is defined inside the function — it gets reset on every call, defeating memoization. Must use `@lru_cache` or define `memo` outside.

**Bug 2:** Longest Common Subsequence: backtracking condition `if dp[i][j] == dp[i-1][j]` picks from "up" before checking the diagonal match, giving a wrong LCS when multiple paths tie.

**Bug 3:** Coin change: `dp[0] = 1` (one way to make 0) is correct, but `dp` initialized to `float('inf')` instead of 0 for non-zero amounts — minimum coins requires initialization to inf, but the code uses 0, which makes every amount look "already reachable."

</details>

## Common Errors & Troubleshooting

| Error | Cause | Fix |
|-------|-------|-----|
| `ModuleNotFoundError: No module named 'X'` | Package not installed | Run `!pip install X` in a cell, then restart kernel |
| `CUDA out of memory` | GPU RAM exceeded | Reduce batch size, or add `torch.cuda.empty_cache()` |
| `RuntimeError: Expected all tensors on same device` | Mixed CPU/GPU tensors | Add `.to(device)` after creating each tensor |
| `ValueError: shapes not aligned` | Matrix dimension mismatch | Print `tensor.shape` before the operation to debug |
| `KeyError` in DataFrame | Column name wrong or missing | Print `df.columns` to see exact column names |
| `IndexError: index out of range` | Loop or slice exceeds sequence length | Print `len(sequence)` and check your index |
| Kernel dies silently | Memory overflow (RAM) | Restart kernel, reduce data size, use generators |
| `UserWarning: No GPU found` | CUDA not available | Add `device = 'cuda' if torch.cuda.is_available() else 'cpu'` |

In [ ]:
# BUG 1: memo is local — memoization doesn't work
def fib_buggy(n):
    memo = {}  # should be defined outside or use @lru_cache
    if n in memo: return memo[n]
    if n <= 1: return n
    memo[n] = fib_buggy(n-1) + fib_buggy(n-2)
    return memo[n]

import time
t0 = time.time()
result = fib_buggy(30)
print(f"fib(30) = {result}, time = {time.time()-t0:.3f}s (should be fast with memoization)")

# BUG 2: LCS backtracking visits wrong path
def lcs_buggy(s1, s2):
    n, m = len(s1), len(s2)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(1,n+1):
        for j in range(1,m+1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    # Backtrack
    i, j = n, m
    lcs = []
    while i>0 and j>0:
        if dp[i][j] == dp[i-1][j]:  # BUG 2: checks up before diagonal
            i -= 1
        elif dp[i][j] == dp[i][j-1]:
            j -= 1
        else:
            lcs.append(s1[i-1])
            i -= 1; j -= 1
    return "".join(reversed(lcs))

print(f"LCS of ABCBDAB, BDCAB: {lcs_buggy('ABCBDAB','BDCAB')}")
print(f"Expected length: 4 (e.g., BCAB or BDAB)")

# BUG 3: coin change wrong initialization
def coin_change_buggy(coins, amount):
    dp = [0] * (amount+1)  # BUG 3: should be [float('inf')] * (amount+1)
    dp[0] = 0
    for a in range(1, amount+1):
        for c in coins:
            if c <= a:
                dp[a] = min(dp[a], dp[a-c]+1)
    return dp[amount] if dp[amount] != float("inf") else -1

print(f"Min coins for 11 with [1,5,6,9]: {coin_change_buggy([1,5,6,9], 11)}")
print(f"Expected: 2 (9+1+1 = 3? No: 5+6=11, so 2 coins)")


## Notebook Complete

**You can now:**
- Apply dynamic programming to sequence alignment, knapsack, and longest-path problems
- Recognise optimal-substructure patterns and implement memoisation/tabulation

**Where this fits:**
- Previous: 01_strings_dna
- Next: 03_graphs_assembly — check the README

**Before moving on, check:**
- [ ] All code cells ran without errors
- [ ] You understand what each function does (read the docstrings)
- [ ] You can explain the key concept in one sentence without looking at notes